# Spatially Constrained Clustering and Epicardial Visualization
## Mouse Embryonic Heart — MOSTA Stereo-seq Dataset

**Associated manuscript:** *(insert citation)*  
**Dataset:** MOSTA E13.5 whole-embryo Stereo-seq (Bin 50; ~25 µm resolution)  
**Source:** PMID 35512705  
**Author:** Quang Dang  

---

## Overview

This notebook reproduces the computational isolation of the embryonic heart from a whole-embryo Stereo-seq dataset and the subsequent spatial co-expression visualization of epicardial marker genes. The pipeline proceeds in three main stages:

1. **Spatially Constrained Clustering (SCC)** — a binary union of an expression-space and a spatial nearest-neighbor graph is used to segment the heart cluster from the whole embryo
2. **Gaussian Smoothing** — localized expression-space smoothing reduces sparsity and reconstructs continuous tissue domains within the heart subset
3. **Bivariate Co-expression Visualization** — custom RGB mapping reveals spatial co-localization of *Upk3b* (epicardial) and *Tbx20* (myocardial/epicardial)

---

## Table of Contents
1. [Imports and Setup](#1-imports-and-setup)
2. [Load Data](#2-load-data)
3. [Whole-Embryo Preprocessing](#3-whole-embryo-preprocessing)
4. [Spatially Constrained Clustering (SCC)](#4-spatially-constrained-clustering)
5. [Heart Subset Extraction](#5-heart-subset-extraction)
6. [Heart-Specific Preprocessing and Smoothing](#6-heart-specific-preprocessing-and-smoothing)
7. [Bivariate Co-expression Visualization](#7-bivariate-co-expression-visualization)

---
## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import scanpy as sc
import squidpy as sq
import stereo as st

sc.settings.verbosity = 1
plt.rcParams['figure.dpi'] = 150

---
## 2. Load Data

The input is a raw-count `.h5ad` file from the MOSTA E13.5 whole-embryo Stereo-seq dataset (sample E1S1). The file is loaded into Scanpy, the raw counts are preserved, and the object is then converted to StereoPy format for spatially-aware processing.

> **Note:** The `bin_type` and `bin_size` attributes must be set manually after conversion to correctly inform downstream smoothing about the physical scale of the data (Bin 50 = 50 × 50 DNBs ≈ 25 µm).

In [ ]:
# ── Update path to match your local environment ────────────────────────────────
H5AD_PATH = '/path/to/E13.5_E1S1_MOSTA.h5ad'

# Load into Scanpy
adata = sc.read(H5AD_PATH)

# Preserve raw counts as a layer, then set .X to raw counts for StereoPy
adata.layers['normalised_counts'] = adata.X.copy()
adata.X = adata.layers['count'].copy()

print(adata)

In [ ]:
# Convert to StereoPy (spatial_key tells it where coordinates live in .obsm)
data = st.io.anndata_to_stereo(adata, spatial_key='spatial')

# Manually specify bin parameters for correct physical scale
# Bin 50 = 50×50 DNBs; ~25 µm per bin (PMID 35512705)
data.bin_type = 'bins'
data.bin_size = 50

print(data)

---
## 3. Whole-Embryo Preprocessing

Standard QC, normalization, and log-transformation are applied to the full embryo object before clustering.

In [ ]:
print('Running QC...')
data.tl.cal_qc()

print('Checkpointing raw counts...')
data.tl.raw_checkpoint()  # Saves raw counts for optional downstream use

print('Normalizing (target sum = 1e4)...')
data.tl.normalize_total(target_sum=1e4)

print('Log1p transforming...')
data.tl.log1p()

print('Done.')

---
## 4. Spatially Constrained Clustering (SCC)

SCC constructs a joint neighborhood graph by taking the **binary union** of:
- An **expression-based** kNN graph (*k* = 30) derived from the top 30 PCs of 2,000 HVGs
- A **spatial** kNN graph (*k* = 8) based on physical coordinates

Using a Boolean OR operation (`conn_expr + conn_spatial > 0`) rather than weighted addition keeps the combined graph sparse and computationally tractable. Fast Leiden clustering (resolution = 1.0, `flavor='igraph'`, `n_iterations=2`) is then applied to the joint graph.

> **Why SCC?** In whole-embryo data, purely expression-based clustering can group spatially distant but transcriptionally similar tissues together. Incorporating spatial adjacency constraints the solution so that clusters are also spatially coherent, enabling clean dissection of individual organs.

In [ ]:
# ── Step 1: Convert back to AnnData for graph construction ────────────────────
adata_temp = st.io.stereo_to_anndata(data, flavor='scanpy')

# ── Step 2: Expression-based kNN graph (k=30, 30 PCs, 2000 HVGs) ─────────────
print('Building expression graph (k=30)...')
sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000)
sc.pp.pca(adata_temp, n_comps=30)
sc.pp.neighbors(
    adata_temp,
    n_neighbors=30,
    n_pcs=30,
    use_rep='X_pca',
    key_added='expression',
    method='umap'  # Uses the optimized UMAP backend
)

# ── Step 3: Spatial kNN graph (k=8) ──────────────────────────────────────────
print('Building spatial graph (k=8)...')
sq.gr.spatial_neighbors(
    adata_temp,
    spatial_key='spatial',
    n_neighs=8,
    coord_type='generic',
    key_added='spatial'
)

print('Done.')

In [ ]:
# ── Step 4: Binary union of connectivity matrices ─────────────────────────────
# Boolean OR: a connection exists if it is present in EITHER the expression
# graph OR the spatial graph. This is preferable to weighted addition because
# it avoids double-counting shared edges and keeps the matrix binary.
print('Computing binary union graph...')
conn_expr    = adata_temp.obsp['expression_connectivities']
conn_spatial = adata_temp.obsp['spatial_connectivities']

conn_joint = (conn_expr + conn_spatial > 0).astype(float)

# Assign joint graph as the active neighbor graph for clustering
adata_temp.obsp['connectivities'] = conn_joint
adata_temp.obsp['distances']      = adata_temp.obsp['expression_distances']
adata_temp.uns['neighbors']       = adata_temp.uns['expression'].copy()

print(f'Joint graph: {conn_joint.shape[0]} cells, {conn_joint.nnz} edges')

In [ ]:
# ── Step 5: Fast Leiden clustering on the joint graph ─────────────────────────
# flavor='igraph'   : Uses the optimized C-core (faster than default leidenalg)
# n_iterations=2    : Limits runtime; avoids over-refinement of small clusters
print(f'Running Leiden clustering ({adata_temp.n_obs} bins)...')
sc.tl.leiden(
    adata_temp,
    resolution=1.0,
    key_added='leiden_scc',
    flavor='igraph',
    n_iterations=2
)

# Store results back into the StereoPy object
cluster_df = adata_temp.obs[['leiden_scc']].rename(columns={'leiden_scc': 'group'})
data.tl.result['leiden_scc'] = cluster_df

print('Cluster counts:')
print(adata_temp.obs['leiden_scc'].value_counts())

In [ ]:
# ── Visualize whole-embryo clusters ──────────────────────────────────────────
# Recenter spatial coordinates so the embryo is at (0,0)
data.position[:, 0] -= data.position[:, 0].min()
data.position[:, 1] -= data.position[:, 1].min()

data.plt.cluster_scatter(
    res_key='leiden_scc',
    dot_size=1,
    show_plotting_scale=False,
    width=10, height=10
)

---
## 5. Heart Subset Extraction

After inspecting the cluster plot, identify the Leiden cluster label corresponding to the heart and update `HEART_LABEL` below. The heart subset is extracted via boolean masking and converted back to AnnData.

In [ ]:
# Inspect available tissue labels if organ-level annotations are available
if 'organ' in data.tl.result:
    print('Available tissue labels:', data.tl.result['organ']['group'].unique())
else:
    print('No organ annotation found — use leiden_scc cluster IDs directly.')
    print('Leiden clusters:', adata_temp.obs['leiden_scc'].unique())

In [ ]:
# ── Update HEART_LABEL to match your cluster/organ annotation ─────────────────
HEART_LABEL = 'Heart'  # or a Leiden cluster ID string e.g. '5'

if 'organ' in data.tl.result:
    subset_mask = data.tl.result['organ']['group'] == HEART_LABEL
else:
    subset_mask = adata_temp.obs['leiden_scc'] == HEART_LABEL

# Bridge to AnnData for robust boolean subsetting
adata_temp_full = st.io.stereo_to_anndata(data, flavor='scanpy')
adata_heart = adata_temp_full[subset_mask].copy()

print(f'Heart subset: {adata_heart.n_obs} bins')

---
## 6. Heart-Specific Preprocessing and Gaussian Smoothing

The isolated heart subset is re-processed independently to capture tissue-specific variance. Highly variable genes and PCs are recalculated. A localized Gaussian smoothing algorithm is then applied, using an expression-space (PCA-based) neighbor graph to aggregate signal across local neighborhoods, reducing sparsity while preserving genuine anatomical boundaries.

In [ ]:
# Convert heart subset to StereoPy
data_heart = st.io.anndata_to_stereo(adata_heart, spatial_key='spatial')
data_heart.bin_type = 'bins'
data_heart.bin_size = 50

print('Preprocessing heart subset...')
data_heart.tl.cal_qc()
data_heart.tl.raw_checkpoint()
data_heart.tl.normalize_total(target_sum=1e4)
data_heart.tl.log1p()

print('Calculating highly variable genes...')
data_heart.tl.highly_variable_genes(
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5,
    res_key='highly_variable_genes',
    n_top_genes=2000
)

print('Running PCA (n=30)...')
data_heart.tl.pca(
    use_highly_genes=True,
    n_pcs=30,
    res_key='pca'
)

print('Done.')

In [ ]:
# ── Gaussian smoothing ────────────────────────────────────────────────────────
# The neighbor graph for smoothing is constructed in PCA expression space
# (not physical space), controlled by pca_res_key.
#
# Parameters:
#   n_neighbors=50       : Number of PCA-space neighbors to aggregate over
#   smooth_threshold=100 : Minimum bin count threshold for smoothing
#   pca_res_key='pca'    : Use the heart-specific PCA embedding

print('Applying Gaussian smoothing (k=50, expression-space neighbors)...')
data_heart.tl.gaussian_smooth(
    n_neighbors=50,
    smooth_threshold=100,
    pca_res_key='pca'
)
print('Smoothing complete. Expression stored in data_heart.exp_matrix.')

In [ ]:
# ── Optional: save checkpoint ─────────────────────────────────────────────────
CHECKPOINT_PATH = '/path/to/heart_subset_bin50_smooth.h5ad'

# st.io.write_h5ad(data_heart, output=CHECKPOINT_PATH)
# print(f'Checkpoint saved to: {CHECKPOINT_PATH}')

# To reload:
# data_heart = st.io.read_h5ad(
#     file_path=CHECKPOINT_PATH,
#     flavor='stereopy',
#     use_raw=True,
#     use_result=True
# )

---
## 7. Bivariate Co-expression Visualization

Spatial co-localization of **Upk3b** (epicardial identity marker) and **Tbx20** (transcription factor expressed highly in the myocardial core but also at low levels in the epicardial lineage) is visualized using a custom bivariate RGB encoding:

| Gene | RGB channels | Resulting color |
|------|-------------|----------------|
| *Upk3b* | R + B | Magenta |
| *Tbx20* | G | Green |
| Co-expression | R + G + B | White / pale magenta |

Because *Tbx20* is expressed at much higher absolute levels in the myocardial core than in the epicardial rim, standard linear scaling would suppress the low-level epicardial signal. An **85th-percentile saturation** strategy is applied independently to each gene: expression values are clipped at the 85th percentile before mapping to [0, 1], effectively normalizing out the hyper-expressing myocardial population and revealing the widespread low-level *Tbx20* in the *Upk3b*-positive epicardial rim.

In this encoding, epicardial cells co-expressing both markers appear as a desaturated magenta or off-white, depending on the relative intensity of both channels.

In [ ]:
def plot_bivariate_coexpression(
    data,
    gene_magenta, gene_green,
    dot_size=10,
    alpha=0.9,
    saturation_percentile=95,
    save_path=None
):
    """
    Bivariate spatial co-expression plot using additive RGB channel mixing.

    Parameters
    ----------
    data                  : StereoPy StereoExpData object after gaussian_smooth
    gene_magenta          : Gene mapped to the magenta channel (R + B)
    gene_green            : Gene mapped to the green channel (G)
    dot_size              : Scatter point size
    alpha                 : Opacity of scatter points
    saturation_percentile : Upper percentile for contrast normalization.
                            Addresses dynamic range discrepancy between cell types.
    save_path             : If provided, saves the figure as SVG

    Color encoding
    --------------
    gene_magenta only  → magenta
    gene_green only    → green
    Both expressed     → pale magenta to white (depending on relative intensity)
    """
    x = data.position[:, 0]
    y = data.position[:, 1]

    def get_norm_vec(gene):
        idx = list(data.gene_names).index(gene)
        vec = data.exp_matrix[:, idx]
        if hasattr(vec, 'toarray'):
            vec = vec.toarray().flatten()
        else:
            vec = vec.flatten()
        cap = np.percentile(vec, saturation_percentile)
        if cap == 0:
            cap = vec.max()
        return np.clip(vec / cap, 0, 1)

    m_norm = get_norm_vec(gene_magenta)
    g_norm = get_norm_vec(gene_green)

    # Only plot bins with at least some expression
    mask = (m_norm > 0) | (g_norm > 0)

    # RGB: magenta = R+B, green = G
    colors = np.stack([m_norm[mask], g_norm[mask], m_norm[mask]], axis=1)

    # Sort by brightness so co-expressing cells are drawn on top
    brightness   = colors.sum(axis=1)
    sort_indices = np.argsort(brightness)
    x_plot  = x[mask][sort_indices]
    y_plot  = y[mask][sort_indices]
    c_plot  = colors[sort_indices]

    # ── Plotting ──────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 8))
    fig.patch.set_facecolor('black')
    gs = gridspec.GridSpec(1, 2, width_ratios=[5, 1], wspace=0.05)

    ax_map = fig.add_subplot(gs[0])
    ax_leg = fig.add_subplot(gs[1])

    # Spatial map
    ax_map.set_facecolor('black')
    ax_map.scatter(x_plot, y_plot, c=c_plot, s=dot_size, alpha=alpha, edgecolors='none')
    ax_map.invert_yaxis()
    ax_map.set_title(
        f'{gene_magenta} (magenta)  +  {gene_green} (green)',
        color='white', fontsize=18
    )
    ax_map.set_aspect('equal')
    ax_map.axis('off')

    # Scale bar (15% of x range)
    bar_len = (x.max() - x.min()) * 0.15
    bar_x   = x.max() - bar_len * 1.2
    bar_y   = y.max() - (y.max() - y.min()) * 0.05
    ax_map.plot([bar_x, bar_x + bar_len], [bar_y, bar_y], color='white', linewidth=4)
    ax_map.text(
        bar_x + bar_len / 2, bar_y - (y.max() - y.min()) * 0.02,
        '500 µm', color='white', ha='center', va='bottom', fontsize=12
    )

    # 2D color legend
    res = 100
    mag_vals, grn_vals = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
    legend_img = np.stack([mag_vals, grn_vals, mag_vals], axis=2)

    ax_leg.imshow(legend_img, origin='lower', extent=[0, 1, 0, 1])
    ax_leg.set_facecolor('black')
    ax_leg.set_xlabel(gene_magenta, color='magenta', fontsize=12, fontweight='bold')
    ax_leg.set_ylabel(gene_green,   color='green',   fontsize=12, fontweight='bold')
    ax_leg.set_xticks([]); ax_leg.set_yticks([])
    ax_leg.set_title('Co-expression\n(White)', color='white', fontsize=10)

    if save_path:
        plt.savefig(
            save_path, format='svg', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor()
        )
        print(f'Saved → {save_path}')

    plt.show()

In [ ]:
# ── Upk3b vs Tbx20: epicardial rim vs myocardial core ─────────────────────────
# saturation_percentile=85 is used here (rather than the default 95) because
# Tbx20 is highly expressed in the myocardium; the lower cap normalizes the
# dynamic range and reveals low-level Tbx20 in the epicardial lineage.
plot_bivariate_coexpression(
    data_heart,
    gene_magenta='Upk3b',
    gene_green='Tbx20',
    dot_size=20,
    alpha=1.0,
    saturation_percentile=85,
    # save_path='figures/upk3b_tbx20_coexpression.svg'
)

---
## Session Info

In [ ]:
import session_info
session_info.show()